# BiRefNet Eğitim Verisi Hazırlığı (Colab)

Bu notebook, `my-bg-remover` projesinin **Faz 2** veri pipeline'ını Google Colab'da
**uçtan uca** çalıştırır: eğitim kaynaklarını (DIS5K, CAMO, COD10K, P3M-10k,
Transparent-460, ...) tam boyutlarıyla indirir, saydamlık/kamuflaj ağırlıklı
kompozit görseller üretir ve sonucu BiRefNet'in resmi eğitim kodunun beklediği
`im/` + `gt/` klasör düzenine dönüştürüp Google Drive'a kaydeder.

**Neden Colab'da?** Lokal geliştirme ortamının disk bütçesi çok kısıtlı (~15GB boş
alan); bu yüzden lokalde yalnız küçük *doğrulama örneklemleri* (`data/raw_train/`,
kaynak başına ≤300MB) indirilip araçlar (script'ler) test edildi. Colab'ın disk
alanı (~200GB, T4/A100 runtime'da) tam veri setlerini indirip işlemeye yeterlidir.

## Ne yapıyor bu notebook? (ML bilmeyenler için özet)

1. **Drive'ı bağla** — Google Drive'ınızı Colab'a bağlıyoruz ki büyük veri Colab'ın
   geçici diskinde kaybolmasın, kalıcı olarak Drive'a yazılsın.
2. **Kod deposunu (repo) indir** — projenin script'lerini (`build_trainset.py`,
   `make_composites.py`, `export_birefnet.py`) Colab'a getiriyoruz.
3. **Ham veriyi indir** — internetten (Hugging Face, Google Drive) eğitim
   görsellerini ve onların "doğru cevap" maskelerini (ground truth / GT — objenin
   tam olarak nerede olduğunu piksel piksel gösteren siyah-beyaz görseller) indiriyoruz.
4. **Manifest oluştur** — hangi görselin hangi kategoriye (saç, saydam, kamuflaj, ...)
   ait olduğunu ve GT dosyasının nerede olduğunu listeleyen bir "envanter" (manifest)
   dosyası üretiyoruz.
5. **Kompozit üret** — objeleri farklı arka planların üzerine yapıştırıp (compositing)
   ve hafif bozulmalar ekleyip (augmentasyon: renk/parlaklık değişimi, JPEG
   bozulması, bulanıklık, ayna görüntü) modelin daha çeşitli örneklerden öğrenmesini
   sağlıyoruz. Saydamlık ve kamuflaj kategorileri, projenin hedefi gereği daha fazla
   kopyayla temsil ediliyor (bkz. hücre (e)).
6. **BiRefNet formatına export et** — eğitimde kullanılacak model (BiRefNet) belirli
   bir klasör düzeni bekliyor (`im/` = görseller, `gt/` = maskeler, aynı dosya adıyla
   eşleşen çiftler). Son adımda veriyi bu düzene dönüştürüyoruz.
7. **Drive'a kopyala + doğrula** — sonucu kalıcı olması için Drive'a kopyalıyor,
   dosya sayılarının tutarlı olduğunu kontrol ediyoruz.

**Bu notebook ÇALIŞTIRILMADAN yazılmıştır** (geliştirme ortamında Colab/GPU/Drive
erişimi yok) — her hücre bağımsız ve açıklamalı olacak şekilde tasarlandı; Colab'da
sırasıyla çalıştırmanız yeterli. Bir hücre hata verirse, hatayı okuyup (çoğunlukla
eksik bir Drive dosyası veya değişen bir HF repo yolu olur) ilgili parametreyi
düzeltip o hücreden devam edebilirsiniz — script'ler idempotent (var olan dosyaları
atlar) tasarlandığı için baştan başlamanıza gerek kalmaz.


## Parametreler

Bu hücredeki değerleri ihtiyacınıza göre değiştirebilirsiniz. İki repo kaynağından
**yalnız birini** doldurun: `REPO_GIT_URL` (projeyi GitHub'a pushladıysanız) veya
`REPO_ZIP_ON_DRIVE` (repo'yu bir zip olarak Drive'a yüklediyseniz — bu ortamda repo
henüz bir GitHub URL'sine sahip olmayabileceğinden pragmatik yedek).


In [ ]:
# --- Repo kaynağı: ikisinden YALNIZ BİRİNİ doldurun ---
REPO_GIT_URL = ""  # ör. "https://github.com/<kullanici>/my-bg-remover.git"
REPO_ZIP_ON_DRIVE = ""  # ör. "/content/drive/MyDrive/my-bg-remover.zip" (git clone yoksa)

# --- Çalışma dizinleri ---
DRIVE_ROOT = "/content/drive/MyDrive"
WORKDIR = "/content/my-bg-remover"          # repo'nun Colab'daki geçici diske açılacağı yer
DRIVE_OUTPUT_SUBDIR = "bg-remover-data"     # sonucun Drive'a yazılacağı klasör adı

# --- Veri parametreleri (Faz 2 planıyla tutarlı varsayılanlar) ---
SEED = 42
CATEGORY_MULTIPLIERS = {"transparent": 4, "camouflage": 2}  # diğer kategoriler x1
TARGET_TOTAL_IMAGES = 40000  # yaklaşık hedef toplam kompozit sayısı (per-image bundan türetilir)
BG_POOL_SIZE = 3000          # arka plan havuzu boyutu (BG-20k'dan; tam 15k mümkün ama disk/süre tasarrufu için alt küme)

# --- Google Drive'dan indirilecek opsiyonel kaynaklar ---
# COD10K-TR zorunlu denenir (drive_id data/train_sources.json'da mevcut, kamuflaj
# hedefi için önemli). HIM2K/AM-2k "general" kategorisine katkı sağlar ama Drive
# erişimi/rıza formu gerektirebilir; bulunamazsa/indirilemezse pipeline BOZULMAZ
# (aşağıdaki hücreler except ile bu durumu tolere eder) — bu yüzden varsayılan
# olarak deneniyor ama isteğe bağlı (İSTEĞE BAĞLI) olarak kapatılabilir.
DOWNLOAD_OPTIONAL_DRIVE_SOURCES = True  # HIM2K + AM-2k denensin mi (False -> atla)

print("Parametreler ayarlandı. REPO_GIT_URL veya REPO_ZIP_ON_DRIVE alanlarından birini doldurduğunuzdan emin olun.")
assert REPO_GIT_URL or REPO_ZIP_ON_DRIVE, "REPO_GIT_URL veya REPO_ZIP_ON_DRIVE alanlarından biri dolu olmalı"


## (a) Google Drive'ı Bağla

Drive'ı bağlamak, Colab'a "bu klasörü benim Google Drive'ımmış gibi kullan" demektir.
Böylece indirdiğimiz/ürettiğimiz veri, Colab oturumu kapandığında (genelde birkaç
saat sonra) SİLİNMEZ — Drive'da kalıcı olarak durur. Çalıştırdığınızda Colab bir
Google hesabı izni isteyecek; onaylayın.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os
os.makedirs(WORKDIR, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/{DRIVE_OUTPUT_SUBDIR}", exist_ok=True)
print(f"Drive bağlandı. Çalışma dizini: {WORKDIR}")
print(f"Çıktı Drive'a şuraya yazılacak: {DRIVE_ROOT}/{DRIVE_OUTPUT_SUBDIR}")


## (b) Repo'yu Getir + Bağımlılıkları Kur

Projenin kodunu (script'ler, `pyproject.toml` bağımlılık listesi) Colab'a getiriyoruz.
Colab'da `uv` kurulu değil (lokal geliştirmede kullanılan hızlı paket yöneticisi) —
bunun yerine standart `pip install -e .` yeterli: `pyproject.toml`'daki
`dependencies` listesini (torch, transformers, pillow, numpy, huggingface-hub, ...)
kurar ve projeyi "editable" (kaynak koddan doğrudan import edilebilir) modda ekler.


In [ ]:
import shutil
import subprocess
import zipfile
from pathlib import Path

if REPO_GIT_URL:
    print(f"git clone {REPO_GIT_URL} -> {WORKDIR}")
    subprocess.run(["git", "clone", REPO_GIT_URL, WORKDIR], check=True)
else:
    print(f"zip açılıyor: {REPO_ZIP_ON_DRIVE} -> {WORKDIR}")
    with zipfile.ZipFile(REPO_ZIP_ON_DRIVE) as zf:
        zf.extractall("/content/_repo_zip_extract")
    # zip'in içinde tek bir üst klasör varsa (ör. "my-bg-remover/") onu WORKDIR'a taşı
    extracted = list(Path("/content/_repo_zip_extract").iterdir())
    src_root = extracted[0] if len(extracted) == 1 and extracted[0].is_dir() else Path("/content/_repo_zip_extract")
    if Path(WORKDIR).exists():
        shutil.rmtree(WORKDIR)
    shutil.move(str(src_root), WORKDIR)

%cd {WORKDIR}
!pip install -e . -q
print("Bağımlılıklar kuruldu.")


## (c) Eğitim Kaynaklarını TAM Olarak İndir

`data/train_sources.json`, Faz 2'de lokalde araştırılmış tüm eğitim kaynaklarının
(nereden indirileceği, lisansı, tahmini boyutu) kaydını tutar. Bu hücre o dosyayı
okuyup her kaynağı **tam** (lokaldeki gibi küçük bir örneklem değil) indirir:

- **Hugging Face kaynakları** (`hf_repo` alanı dolu olanlar: DIS5K-TR, CAMO-TR,
  P3M-10k, Transparent-460, BG-20k): `huggingface_hub.snapshot_download` veya
  parquet parçalarını okuyarak indirilir.
- **Google Drive kaynakları** (`drive_id` alanı dolu olanlar: COD10K-TR zorunlu;
  HIM2K/AM-2k `DOWNLOAD_OPTIONAL_DRIVE_SOURCES` açıksa denenir): `gdown` ile
  indirilir. Bu ortamlarda (Colab) Drive erişimi genelde çalışır; lokal geliştirme
  ortamında bu mümkün değildi (bkz. `data/train_sources.json` notları).
- **Distinctions-646**: hiçbir genel erişimli indirme linki yok (yalnız yazarla
  e-posta ile talep) — bu kaynak HER ZAMAN atlanır, yalnız bir not yazdırılır.

Sonuç, lokal `scripts/build_trainset.py`'nin beklediği ile AYNI dizin yapısına
(`data/raw_train/<kaynak>/{im,gt}` veya kaynağa özgü alt klasörler) yazılır — böylece
bir sonraki hücre (d) aynı script fonksiyonlarını (yalnız tam sayılarla) çağırabilir.


In [ ]:
!pip install gdown -q

import json
from pathlib import Path

RAW = Path("data/raw_train")
RAW.mkdir(parents=True, exist_ok=True)

with open("data/train_sources.json") as f:
    SOURCE_DEFS = {s["name"]: s for s in json.load(f)["sources"]}

print(f"{len(SOURCE_DEFS)} kaynak tanımı yüklendi: {list(SOURCE_DEFS)}")


In [ ]:
# --- Hugging Face kaynakları: TAM split indirilir (parquet parçaları üzerinden) ---
import io
import re

import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem
from PIL import Image

fs = HfFileSystem()


def _sanitize_stem(name) -> str:
    """Parquet'teki kaynak dosya adını güvenli bir stem'e çevirir (uzantıyı at,
    dosya sistemine uygun olmayan karakterleri '_' yap)."""
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", Path(str(name)).stem)


def _download_hf_parquet_pairs(source_name: str, img_col: str, mask_col: str, out_subdir: str) -> int:
    """source_name'in TÜM parquet parçalarını (split_patterns) satır satır okuyup
    (image, mask) çiftlerini out_subdir/{im,gt}/ altına yazar. Bellek dostu: her
    parça ayrı ayrı okunur, tek seferde tüm veri seti belleğe alınmaz.

    Stem stratejisi (ÇAKIŞMA ÖNLEME): şemada `image_name` kolonu varsa stem oradan
    türetilir (kaynak veri setinin kendi benzersiz dosya adları). Yoksa TÜM parquet
    parçaları boyunca artan KÜMÜLATİF bir sayaç kullanılır — parça başına sıfırlanan
    bir indeks, 2. parçadan itibaren 1. parçanın stem'leriyle çakışıp o satırların
    sessizce 'zaten indirilmiş' sanılarak atlanmasına yol açardı (ör. 6 parçalı
    DIS5K-TR'nin yalnız ~1/6'sı inerdi)."""
    spec = SOURCE_DEFS[source_name]
    repo = spec["hf_repo"]
    out_im = RAW / out_subdir / "im"
    out_gt = RAW / out_subdir / "gt"
    out_im.mkdir(parents=True, exist_ok=True)
    out_gt.mkdir(parents=True, exist_ok=True)

    def _bytes_of(cell_value):
        # HF 'Image' feature parquet export'unda değer genelde {"bytes": ..., "path": ...}
        # struct'ı olarak gelir; bazı mirror'larda doğrudan ham byte olabilir.
        return cell_value["bytes"] if isinstance(cell_value, dict) else cell_value

    written = 0
    counter = 0  # kümülatif satır sayacı — parça sınırlarında SIFIRLANMAZ
    for pattern in spec["split_patterns"]:
        paths = fs.glob(f"datasets/{repo}/{pattern}")
        for p in sorted(paths):
            print(f"  okunuyor: {p}")
            with fs.open(p, "rb") as fh:
                schema_names = pq.read_schema(fh).names
                fh.seek(0)
                has_name = "image_name" in schema_names
                columns = (["image_name"] if has_name else []) + [img_col, mask_col]
                table = pq.read_table(fh, columns=columns)
            for i in range(table.num_rows):
                if has_name:
                    stem = f"{source_name}_{_sanitize_stem(table['image_name'][i].as_py())}"
                else:
                    stem = f"{source_name}_{counter:06d}"
                counter += 1
                out_img_path = out_im / f"{stem}.jpg"
                out_gt_path = out_gt / f"{stem}.png"
                if out_img_path.exists() and out_gt_path.exists():
                    continue  # idempotent: zaten indirilmiş (sorted() -> parça sırası stabil, stem'ler deterministik)
                img_bytes = _bytes_of(table[img_col][i].as_py())
                mask_bytes = _bytes_of(table[mask_col][i].as_py())
                Image.open(io.BytesIO(img_bytes)).convert("RGB").save(out_img_path, quality=95)
                Image.open(io.BytesIO(mask_bytes)).convert("L").save(out_gt_path)
                written += 1

    # --- İndirme sonrası bütünlük kontrolü: diskteki çift sayısı beklenen tam set boyutuyla ---
    total_pairs = len(list(out_im.glob("*")))
    expected = spec.get("full_pair_count")
    print(f"{source_name}: {written} yeni çift yazıldı; diskte toplam {total_pairs} (beklenen ~{expected})")
    if expected and total_pairs < 0.9 * expected:
        raise RuntimeError(
            f"{source_name}: diskte yalnız {total_pairs}/{expected} çift var (<%90) — "
            f"stem çakışması, eksik parquet parçası veya değişen repo şeması olabilir; "
            f"yukarıdaki 'okunuyor' loglarını inceleyin."
        )
    return written


# NOT: Yukarıdaki genel fonksiyon parquet şemasındaki kolon adlarına duyarlıdır;
# `nobg/DIS5K` ve `nobg/camo` için Faz 2 lokal örneklem koduyla (scripts/build_trainset.py
# docstring'i) doğrulanmış kolon adları kullanılır. HF repo şeması değişmişse
# `pq.read_schema(...)` çıktısını inceleyip img_col/mask_col'u güncelleyin.
try:
    _download_hf_parquet_pairs("dis5k_tr", img_col="image", mask_col="label", out_subdir="dis5k")
except Exception as e:
    print(f"UYARI: dis5k_tr indirilemedi ({e}); data/raw_train/dis5k mevcutsa (lokal örneklem) o kullanılacak.")

try:
    _download_hf_parquet_pairs("camo_tr", img_col="image", mask_col="mask", out_subdir="camo")
except Exception as e:
    print(f"UYARI: camo_tr indirilemedi ({e}); data/raw_train/camo mevcutsa (lokal örneklem) o kullanılacak.")


In [ ]:
# --- P3M-10k (tek zip) ve Transparent-460 (klasör bazlı) — snapshot_download ile TAM indirme ---
from huggingface_hub import hf_hub_download, snapshot_download

# P3M-10k: tüm veri seti tek bir zip (data/p3m10k.zip, ~5.8GB); indirip TRAIN alt
# kümesini (blurred_image + mask) çıkarıyoruz.
p3m_zip = hf_hub_download(repo_id=SOURCE_DEFS["p3m_10k_train"]["hf_repo"],
                           repo_type="dataset", filename="data/p3m10k.zip")
import zipfile

p3m_out_im = RAW / "p3m" / "im"
p3m_out_gt = RAW / "p3m" / "gt"
p3m_out_im.mkdir(parents=True, exist_ok=True)
p3m_out_gt.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(p3m_zip) as zf:
    names = [n for n in zf.namelist() if "/train/blurred_image/" in n or "/train/mask/" in n]
    for n in names:
        stem = Path(n).stem
        target = (p3m_out_im if "/blurred_image/" in n else p3m_out_gt) / Path(n).name
        if target.exists():
            continue
        with zf.open(n) as src, open(target, "wb") as dst:
            dst.write(src.read())
print(f"p3m_10k_train: {len(list(p3m_out_im.iterdir()))} görsel -> {p3m_out_im.parent}")

# Transparent-460: Train/{fg,alpha}/ tam split (410 çift, orijinal çözünürlük).
import shutil

trans_dir = snapshot_download(repo_id=SOURCE_DEFS["transparent_460_train"]["hf_repo"],
                               repo_type="dataset", allow_patterns=["Train/*"])
trans_out = RAW / "trans460_train"
if trans_out.exists():
    shutil.rmtree(trans_out)
shutil.copytree(Path(trans_dir) / "Train" / "fg", trans_out / "fg")
shutil.copytree(Path(trans_dir) / "Train" / "alpha", trans_out / "alpha")
print(f"transparent_460_train: {len(list((trans_out / 'fg').iterdir()))} görsel -> {trans_out}")


In [ ]:
# --- Google Drive kaynakları: gdown ile indirme ---
import zipfile as _zf


def _gdown_extract(drive_id: str, out_dir: Path, label: str) -> bool:
    """Drive id'sinden zip indirip out_dir'e açar. Başarısız olursa False döner
    (pipeline'ı durdurmaz — çağıran taraf notla geçer)."""
    out_dir.mkdir(parents=True, exist_ok=True)
    zip_path = out_dir.parent / f"{out_dir.name}.zip"
    try:
        import gdown

        gdown.download(id=drive_id, output=str(zip_path), quiet=False)
        with _zf.ZipFile(zip_path) as zf:
            zf.extractall(out_dir)
        print(f"{label}: indirildi ve açıldı -> {out_dir}")
        return True
    except Exception as e:
        print(f"UYARI: {label} indirilemedi ({e}) — bu kaynak ATLANACAK (görev talimatı: bulunamazsa/erişilemezse atla).")
        return False


# COD10K-TR: kamuflaj hedefi için önemli, zorunlu denenir.
cod10k_ok = _gdown_extract(SOURCE_DEFS["cod10k_tr"]["drive_id"], RAW / "cod10k_raw", "COD10K-TR")

# HIM2K / AM-2k: opsiyonel (DOWNLOAD_OPTIONAL_DRIVE_SOURCES bayrağına bağlı).
him2k_ok = am2k_ok = False
if DOWNLOAD_OPTIONAL_DRIVE_SOURCES:
    him2k_ok = _gdown_extract(SOURCE_DEFS["him2k"]["drive_id"], RAW / "him2k_raw", "HIM2K")
    am2k_ok = _gdown_extract(SOURCE_DEFS["am_2k"]["drive_id"], RAW / "am2k_raw", "AM-2k")
else:
    print("HIM2K/AM-2k atlandı (DOWNLOAD_OPTIONAL_DRIVE_SOURCES=False).")

print(f"\nDistinctions-646: {SOURCE_DEFS['distinctions_646']['download_method']}")
print("-> HER ZAMAN atlanır (genel erişimli indirme linki yok).")

print(f"\nÖzet: COD10K-TR={cod10k_ok}, HIM2K={him2k_ok}, AM-2k={am2k_ok}")
print("NOT: COD10K-TR/HIM2K/AM-2k zip içeriklerinin gerçek klasör yapısı resmi "
      "dağıtıma göre değişebilir (Drive üzerinden barındırılan setler HF gibi "
      "standart bir şema garantilemez) — indirme başarılıysa bir sonraki hücrede "
      "(d) bu klasörleri build_trainset ile eşlemeden önce `!find data/raw_train/cod10k_raw -maxdepth 3` "
      "gibi bir komutla gerçek alt klasör adlarını (image/ vs Image/ vs img/, GT/ vs mask/) "
      "kontrol edip aşağıdaki glob'ları buna göre güncelleyin.")


## (d) Tam Manifest Üret (`build_trainset.py` mantığıyla)

Şimdi indirdiğimiz ham veriden `data/train/manifest.jsonl` dosyasını üretiyoruz —
bu, her eğitim örneğinin kimliğini (id), görsel yolunu, kategorisini ve GT maske
yolunu tutan bir "envanter" dosyasıdır (proje genelinde test setiyle de aynı format
kullanılır, bkz. `benchmark/testset.py`).

Kaynak tanımları (glob desenleri + kategori kuralları) `scripts/build_trainset.py`
içindeki **`SOURCE_SPECS`** sözlüğünden okunur — lokal script'le notebook'un aynı
tanımı paylaştığı TEK doğruluk kaynağı; elle kopyalanmış glob'ların zamanla sapması
(drift) böyle önlenir. Lokal koşuda kullanılan küçük örneklem boyutları
(`LOCAL_SAMPLE_N`, disk bütçesi kısıtı) burada KULLANILMAZ: aynı fonksiyonları
(`sample_source`, `sample_disvd_tokens`, `copy=True` modu — sembolik bağlantı yerine
gerçek dosya, çünkü Drive'a taşırken symlink kırılabilir) **tam setle** (`n=None` ->
tüm eşleşen çiftler) çağırıyoruz, artı Google Drive'dan gelen COD10K-TR (ve varsa
HIM2K/AM-2k) kaynaklarını ekliyoruz.


In [ ]:
import sys

sys.path.insert(0, "scripts")
import build_trainset as bt  # noqa: E402
from benchmark.testset import append_entries  # noqa: E402

# TEK DOĞRULUK KAYNAĞI: glob/kategori tanımları build_trainset.SOURCE_SPECS'ten
# okunur (elle kopya YOK — lokal script değişirse notebook otomatik uyumlu kalır).
# Yalnız n=None (tam set) ve copy=True (Colab'da gerçek dosya, symlink değil) farkı var.
FULL_SOURCES = [
    (name, spec["img_glob"], spec["gt_glob"], spec["category"], None)
    for name, spec in bt.SOURCE_SPECS.items()
    if spec["category"] != "disvd_tokens"  # dis5ktr aşağıda sample_disvd_tokens ile işlenir
]

# COD10K-TR indirildiyse ekle (gerçek alt klasör adları (c) hücresindeki NOT'a göre
# doğrulanmalı; burada yaygın SINet dağıtım şeması varsayılıyor: Image/ + GT/).
if cod10k_ok:
    FULL_SOURCES.append(
        ("cod10ktr", "data/raw_train/cod10k_raw/**/Image/*", "data/raw_train/cod10k_raw/**/GT/*",
         "camouflage", None)
    )
if him2k_ok:
    FULL_SOURCES.append(
        ("him2k", "data/raw_train/him2k_raw/**/*.jpg", "data/raw_train/him2k_raw/**/*_matte.png",
         "general", None)
    )
if am2k_ok:
    FULL_SOURCES.append(
        ("am2k", "data/raw_train/am2k_raw/**/original/*", "data/raw_train/am2k_raw/**/mask/*",
         "general", None)
    )

bt.OUT_IMG.mkdir(parents=True, exist_ok=True)
bt.OUT_GT.mkdir(parents=True, exist_ok=True)
total_rows = 0
for src in FULL_SOURCES:
    rows = bt.sample_source(*src, copy=True)
    if rows:
        append_entries(str(bt.MANIFEST), rows)
    total_rows += len(rows)
    print(f"{src[0]} ({src[3]}): {len(rows)} çift eklendi")

# DIS5K-TR: tek havuzdan örneklenir, kategori dosya adı token'ından (thin/complex) atanır.
dis_spec = bt.SOURCE_SPECS["dis5ktr"]
dis_rows = bt.sample_disvd_tokens("dis5ktr", dis_spec["img_glob"], dis_spec["gt_glob"],
                                   n=None, copy=True)
if dis_rows:
    append_entries(str(bt.MANIFEST), dis_rows)
total_rows += len(dis_rows)

print(f"\nToplam manifest satırı (bu koşuda eklenen): {total_rows}")
print(f"Manifest: {bt.MANIFEST}")


## (e) Kompozit + Augmentasyon Üretimi (`make_composites.py`)

Bu adım her eğitim görselini bir arka planla birleştirir (compositing) ve hafif
bozulmalar ekler (augmentasyon). Kategori bazlı çarpanlar (`CATEGORY_MULTIPLIERS`,
parametre hücresinde tanımlı) saydamlık ve kamuflaj kategorilerinin eğitim
karışımında daha yüksek payla temsil edilmesini sağlar — projenin hedefi bu iki
kategoride güçlü performans olduğundan. Kamuflaj kategorisinde **compositing
uygulanmaz** (yalnız augment): kamuflajın özü obje ile orijinal arka planının doku/
renk uyumu olduğundan, rastgele bir arka plana yapıştırmak bu sinyali yok eder
(bkz. `bgr/compositing.py` ve Faz 2 planı Task 4 notu).

Önce arka plan havuzunu (BG-20k'dan `BG_POOL_SIZE` kadar görsel) indiriyoruz, sonra
`make_composites.run(...)` fonksiyonunu (lokal örneklemde test edilmiş AYNI fonksiyon)
çağırıyoruz. `per_image` değeri, `TARGET_TOTAL_IMAGES` hedefinden kabaca geriye
hesaplanır (kesin değil — kategori çarpanları nedeniyle tam sayı tutturmak zor;
yaklaşık bir tahmindir).


In [ ]:
# --- Arka plan havuzu: BG-20k'dan BG_POOL_SIZE kadar görsel ---
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem
from PIL import Image
import io

bg_dir = Path("data/backgrounds")
bg_dir.mkdir(parents=True, exist_ok=True)
existing_bg = len(list(bg_dir.iterdir()))
if existing_bg >= BG_POOL_SIZE:
    print(f"data/backgrounds zaten {existing_bg} görsel içeriyor (>= {BG_POOL_SIZE}); indirme atlanıyor.")
else:
    fs = HfFileSystem()
    bg_spec = SOURCE_DEFS["bg_20k"]
    pattern = bg_spec["split_patterns"][0]  # "data/train-*-of-00022.parquet"
    parts = sorted(fs.glob(f"datasets/{bg_spec['hf_repo']}/{pattern}"))
    written = existing_bg
    for part in parts:
        if written >= BG_POOL_SIZE:
            break
        with fs.open(part, "rb") as fh:
            table = pq.read_table(fh, columns=["image"])
        for i in range(table.num_rows):
            if written >= BG_POOL_SIZE:
                break
            out_path = bg_dir / f"bg20k_{written:06d}.jpg"
            if out_path.exists():
                written += 1
                continue
            img_bytes = table["image"][i].as_py()["bytes"]
            im = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            im.thumbnail((1024, 1024))
            im.save(out_path, format="JPEG", quality=88)
            written += 1
    print(f"data/backgrounds: {written} arka plan görseli.")


In [ ]:
# --- per_image tahmini: hedef toplam / (kaynak satırı sayısı * ortalama çarpan) ---
import make_composites as mc  # scripts/ zaten sys.path'te (bkz. hücre d)

mc.CATEGORY_MULTIPLIER = dict(CATEGORY_MULTIPLIERS)  # parametre hücresinden gelen çarpanlar

from benchmark.testset import load_manifest

manifest_rows = [r for r in load_manifest(str(bt.MANIFEST)) if r.get("gt_alpha")]
avg_multiplier = sum(mc.multiplier(r["category"]) for r in manifest_rows) / max(len(manifest_rows), 1)
per_image = max(1, round(TARGET_TOTAL_IMAGES / (len(manifest_rows) * avg_multiplier))) if manifest_rows else 1
print(f"Kaynak satırı: {len(manifest_rows)}, ortalama çarpan: {avg_multiplier:.2f}, "
      f"seçilen per_image: {per_image} (tahmini toplam: {len(manifest_rows) * avg_multiplier * per_image:.0f})")

counts = mc.run(
    manifest_path=bt.MANIFEST,
    backgrounds_dir=bg_dir,
    per_image=per_image,
    seed=SEED,
    out_dir="data/train_composites",
)
print("Kategori bazlı üretilen kompozit sayısı:", counts)


## (f) BiRefNet Formatına Export (`export_birefnet.py`)

Son adım: `data/train_composites/manifest.jsonl`'i BiRefNet'in resmi eğitim kodunun
beklediği düzene (`TRAIN/im/*.jpg` + `TRAIN/gt/*.png`, aynı dosya adıyla eşleşen
çiftler) dönüştürüyoruz; ayrıca `stats.json` (toplam sayı, kategori dağılımı,
çözünürlük yüzdelikleri, kategori bazında "yumuşak alfa" oranı — saydamlık/matting
setlerinde kısmi şeffaf piksellerin payı) yazılıyor.


In [ ]:
import json

import export_birefnet as eb  # scripts/ zaten sys.path'te

stats = eb.export(
    manifest_path="data/train_composites/manifest.jsonl",
    out_dir="/content/birefnet_format",
    split_name="TRAIN",
)
print(json.dumps(stats, ensure_ascii=False, indent=2))


## (g) Drive'a Kopyala + Bütünlük Kontrolü

Son olarak `/content/birefnet_format` (Colab'ın geçici diski — oturum kapanınca
silinir) klasörünü Drive'daki kalıcı konuma (`bg-remover-data/`) kopyalıyoruz.
Kopyalama sonrası dosya sayılarının (kaynak vs. hedef, `im/` vs `gt/`) birebir
eşleştiğini ve `stats.json`'daki toplam sayıyla tutarlı olduğunu doğruluyoruz —
Drive senkronizasyonu bazen büyük klasürlerde yarıda kesilebildiğinden bu kontrol
önemlidir.


In [ ]:
import json
import shutil
from pathlib import Path

src = Path("/content/birefnet_format")
dst = Path(DRIVE_ROOT) / DRIVE_OUTPUT_SUBDIR

print(f"Kopyalanıyor: {src} -> {dst}")
shutil.copytree(src, dst, dirs_exist_ok=True)

# --- Bütünlük kontrolü ---
src_im = list((src / "TRAIN" / "im").iterdir())
src_gt = list((src / "TRAIN" / "gt").iterdir())
dst_im = list((dst / "TRAIN" / "im").iterdir())
dst_gt = list((dst / "TRAIN" / "gt").iterdir())

with open(src / "stats.json") as f:
    stats_on_disk = json.load(f)

print(f"im/: kaynak={len(src_im)}, hedef={len(dst_im)}")
print(f"gt/: kaynak={len(src_gt)}, hedef={len(dst_gt)}")
print(f"stats.json total: {stats_on_disk['total']}")

assert len(src_im) == len(dst_im), "im/ dosya sayısı Drive kopyasında eşleşmiyor!"
assert len(src_gt) == len(dst_gt), "gt/ dosya sayısı Drive kopyasında eşleşmiyor!"
assert len(dst_im) == len(dst_gt) == stats_on_disk["total"], "im/gt/stats.json toplam sayıları tutarsız!"

print("\nBÜTÜNLÜK KONTROLÜ BAŞARILI — veri Drive'da hazır.")
print(json.dumps(stats_on_disk, ensure_ascii=False, indent=2))
